In [0]:
import requests
import pandas as pd

from pyspark.sql.functions import col, current_timestamp, lit
from delta.tables import DeltaTable

In [0]:
# Função para obter a última rodada processada
def get_ultima_rodada_processada():
    try:
        # Consulta a maior rodada registrada na tabela
        result = spark.sql("""
            SELECT MAX(rodada) as ultima
            FROM project_data_football_bronze.partidas_rodada
        """).collect()[0]["ultima"]

        # Retorna a rodada ou 0 se não houver registros
        return result if result is not None else 0

    except:
        # Retorna 0 caso a tabela não exista ou ocorra erro
        return 0

In [0]:
# Obtém o status do mercado da API
status_mercado = requests.get(
    "https://api.cartola.globo.com/mercado/status"
).json()

# Recupera a rodada atual da API
rodada_atual = status_mercado["rodada_atual"]

# Recupera a última rodada processada na tabela
ultima_processada = get_ultima_rodada_processada()

print("Última rodada processada:", ultima_processada)
print("Rodada atual da API:", rodada_atual)

# Gera a lista de rodadas para processar
rodadas_para_processar = range(1, rodada_atual + 1)

print("Rodadas que serão garantidas no MERGE:", list(rodadas_para_processar))

In [0]:
for rodada in rodadas_para_processar:

    # Inicia ingestão da rodada
    print(f"Iniciando ingestão da rodada {rodada}")

    # Monta URL e obtém partidas da rodada
    url_partidas = f"https://api.cartola.globo.com/partidas/{rodada}"
    partidas_json = requests.get(url_partidas).json()

    partidas = partidas_json.get("partidas", [])

    # Verifica se há partidas
    if not partidas:
        print(f"Nenhuma partida encontrada para rodada {rodada}")
        continue

    # Cria DataFrame Spark das partidas
    df_partidas = spark.createDataFrame(pd.DataFrame(partidas))

    # Ajusta tipos e adiciona colunas
    df_partidas = (
        df_partidas
        .withColumn("placar_oficial_mandante", col("placar_oficial_mandante").cast("int"))
        .withColumn("placar_oficial_visitante", col("placar_oficial_visitante").cast("int"))
        .withColumn("rodada", lit(rodada))
        .withColumn("dt_ingestao", current_timestamp())
    )

    tabela = "project_data_football_bronze.partidas_rodada"

    # Verifica se a tabela existe
    if spark.catalog.tableExists(tabela):

        # Faz merge dos dados na tabela Delta
        delta_table = DeltaTable.forName(spark, tabela)

        (
            delta_table.alias("target")
            .merge(
                df_partidas.alias("source"),
                "target.partida_id = source.partida_id"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

        print(f"Rodada {rodada} mesclada com sucesso")

    else:

        # Cria tabela Delta se não existir
        df_partidas.write \
            .format("delta") \
            .mode("overwrite") \
            .saveAsTable(tabela)

        print(f"Tabela criada e rodada {rodada} carregada")

print("Ingestão finalizada com sucesso.")